# Record to S3, index, and search your camera

Run these numbered steps with **Shift+Enter**, as in HomeSight Core's `ExampleTrial.ipynb`. This tutorial records a live camera for 60 seconds, inspects the shared catalog, runs a finite indexing job directly, and searches the retained footage. Uploaded clips and local state are retained.

## 1. Start the test Qdrant interface first

From the repository root, create `.env` from `.env.example` if needed and replace the placeholders. Run:

```sh
docker compose --env-file .env config --quiet
docker compose up -d qdrant
uv sync --group notebooks --extra s3
uv run --group notebooks --extra s3 jupyter lab notebooks/ExampleTrial.ipynb
```

Open http://localhost:6333/dashboard (or your `QDRANT_HTTP_PORT`) and enter `HOMESIGHT_QDRANT_API_KEY`. Select this repository's `.venv` kernel. Qdrant uses a persistent named volume and binds only to this host. No task queue is used.

The camera, existing S3 bucket, and compatible embedding model must be available for the live steps. For a service-free check first, run `uv run docs/examples/offline.py`.


In [ ]:
"""Run the live analytics tutorial using installed package APIs."""

import threading
import time
from contextlib import closing
from pathlib import Path

from IPython.display import Video, display
from qdrant_client import QdrantClient

from homesight_analytics import (
    CoreSegmentStorageReader,
    HomesightConfig,
    IndexingRunReport,
    SampledFrames,
    SegmentCatalog,
    SegmentRecord,
    SegmentSearchResult,
    StreamStatus,
    create_recording_service_from_config,
    load_homesight_config,
    load_homesight_from_config_file,
)

working_directory: Path = Path.cwd().resolve()
project_root: Path = (
    working_directory
    if (working_directory / "src/homesight_analytics").is_dir()
    else working_directory.parent
)
if not (project_root / "src/homesight_analytics").is_dir():
    raise RuntimeError("Launch from the repository root or notebooks/ directory.")
config_path: Path = project_root / "configs/tutorial.toml"
env_path: Path = project_root / ".env"
print("Repository:", project_root)

## 2. Load the shared S3 and Qdrant settings

Keep these in the root `.env`:

- Camera and **S3 settings**: `TEST_RTSP_URL`, `S3_ENDPOINT`, `S3_BUCKET`, `S3_ACCESS_KEY`, `S3_SECRET_KEY`, and `S3_SESSION_TOKEN` (present but empty if unused).
- Qdrant: `QDRANT_HOST`, `QDRANT_HTTP_PORT`, `QDRANT_GRPC_PORT`, `QDRANT_COLLECTION`, and `HOMESIGHT_QDRANT_API_KEY`.
- Model: `HOMESIGHT_MODEL_BASE_URL`, `HOMESIGHT_MODEL_NAME`, `HOMESIGHT_EMBEDDING_DIMENSION`, and `HOMESIGHT_MODEL_API_KEY`.

Reuse the S3 values from your Core setup under the `S3_*` names above. Its S3/MinIO server and bucket remain separate from this Qdrant Compose service. The tutorial uses path-style S3 addressing, TLS verification, and region `us-east-1`; edit `configs/tutorial.toml` if your storage differs. With HomeSight Core v0.1.1, the tutorial samples at **2 fps** and sends the complete chronological JPEG sequence together in one segment prompt. Actual frame counts depend on footage duration and available source frames, including partial final seconds; frames are never duplicated or padded. The endpoint must accept the complete sequence and return **one shared-space embedding per segment**, alongside compatible text embeddings. Check its image-count and request-size limits before indexing; the library never silently truncates frames.

The tutorial input settings are:

```toml
[embedding_backend.input]
segment_input_mode = "frames"
transport = "data_url"
frames_per_second = 2
frame_width = 448
frame_height = 448
```

A positive integer rate overrides `frame_count`. Omit it to use uniform fixed-count sampling (`frame_count = 8` by default). When changing model or preprocessing, including switching from eight fixed frames, choose a fresh `embedding_backend.model_identifier` in the TOML and a fresh `QDRANT_COLLECTION` in `.env`. Stop old analytics workers, load the updated components, call `components.catalog.mark_segments_for_reindexing()`, and rerun step 6 to reindex retained footage. The catalog schema and search API stay the same.

Process variables override dotenv values. Secrets stay on validated models; do not display the full configuration. Paths resolve against the TOML directory. This cell does not connect to any service.


In [ ]:
config: HomesightConfig = load_homesight_config(config_path, env_file=env_path)
camera_name: str = config.cameras[0].camera_name
catalog = SegmentCatalog(config.catalog.database_path)
print("Configuration loaded for camera:", camera_name)
print("Catalog:", config.catalog.database_path)
print("Collection:", config.vector_store.collection_name)

## 3. Check Qdrant before recording

This reads the test server's collection list using the configured key. It does not create a collection yet. A connection or authentication failure here means the Compose service or `.env` settings need attention. S3 objects are not stored in Qdrant; it stores vectors and metadata.


In [ ]:
with closing(
    QdrantClient(
        host=config.vector_store.host,
        port=config.vector_store.http_port,
        grpc_port=config.vector_store.grpc_port,
        prefer_grpc=config.vector_store.prefer_grpc,
        https=config.vector_store.use_https,
        api_key=(
            config.vector_store.api_key.get_secret_value() if config.vector_store.api_key else None
        ),
    )
) as qdrant:
    print("Qdrant reachable; collections:", len(qdrant.get_collections().collections))

## 4. Record a bounded live session to S3

**Running this cell starts your camera and uploads clips.** It records for 60 seconds plus shutdown time. Motion warm-up defaults to 30 seconds; unknown results remain eligible for indexing. Confirmed no-motion clips are retained but skipped by the default indexing policy.

The recording service commits and acknowledges events continuously. Interrupting the cell closes the service through its context manager. Staging and the catalog remain under `data/tutorial/`; uploaded objects remain under `homesight-analytics-tutorial`. Repeating this cell resumes unfinished work and records new footage with the same settings.

Stopping preserves unfinished analysis/uploads rather than draining all backlog. A pending upload is not a successful upload. No model or Qdrant connection is needed during recording.


In [ ]:
RECORD_SECONDS = 60.0
if RECORD_SECONDS <= 0:
    raise ValueError("Recording duration must be positive.")
stop = threading.Event()
started: float = time.monotonic()
with create_recording_service_from_config(config, catalog) as recording:
    print("Recording started; interrupt this cell to stop early.")
    while not stop.wait(1):
        recording.check_service_health()
        if time.monotonic() - started >= RECORD_SECONDS:
            break
print("Recording stopped and emitted events committed; unfinished jobs remain resumable.")

## 5. Inspect metadata and storage readiness

Times are aware UTC values. Duration comes from media; keyframes can make it exceed the target ten seconds. `effective_motion` includes any later reanalysis override; the original source result remains in `motion`. Unknown decisions use `has_motion=None`, distinct from a valid negative result.

Only available, retained archives can be indexed. If uploads remain pending, check the camera/S3 service and repeat step 4 to resume work; that also records additional footage. Saved recording status is a timestamped observation, not proof that a recorder is currently alive.


In [ ]:
status: StreamStatus = catalog.get_stream_status(camera_name)
print(status.model_dump_json(indent=2))
records: list[SegmentRecord] = catalog.list_segments([camera_name])
for segment in records[:10]:
    print(
        segment.segment_id,
        segment.start_time.isoformat(),
        segment.duration_seconds,
        segment.video_codec,
        segment.storage_status,
        "motion:",
        segment.effective_motion.status,
        segment.effective_motion.has_motion,
    )
print("Retained available segments:", sum(item.is_available for item in records))

### Inspect the sampled frames and timestamps

This optional cell decodes the first available archive through the same pinned local-file cache used by indexing. It makes no model request. `SampledFrames.timestamps_ms` contains actual clip-relative presentation times aligned with the RGB frame axis. JPEG and tensor encoders retain these times in `EncodedInput.timestamps_ms`; default HTTP requests send only the images. CPU decoding and the native ten-second timeout remain the defaults. Native output limits (4096 frames or 1 GiB of RGB pixels) fail explicitly.


In [ ]:
available: list[SegmentRecord] = [item for item in records if item.is_available]
rate: int | None = config.embedding_backend.input.frames_per_second
if available and rate is not None:
    reader = CoreSegmentStorageReader(
        config.storage,
        local_cache_directory=config.cache.local_cache_directory,
        local_cache_max_bytes=config.cache.local_cache_max_bytes,
    )
    try:
        with reader.open_segment_media(available[0].storage_uri or "") as media:
            sampled: SampledFrames = media.sample_frames_at_rate(
                rate,
                config.embedding_backend.input.frame_width,
                config.embedding_backend.input.frame_height,
            )
        print("RGB frame shape:", sampled.frames.shape, sampled.frames.dtype)
        print("Clip-relative timestamps (ms):", sampled.timestamps_ms)
    finally:
        reader.close()
else:
    print("Record available footage and select frames_per_second first.")

## 6. Run one finite indexing job

This opens the configured embedding transport and validates/creates the model-owned Qdrant collection. It sends every sampled frame from a segment in one prompt, validates exactly one returned embedding, and stores one Qdrant point for that segment. It does not average separate frame vectors. It processes a snapshot sequentially, then returns; it does not start recording or a scheduler. Later arrivals wait for another call. A zero count can mean pending uploads, confirmed no-motion footage, or already indexed segments.

Failures are persisted and later segments continue. Inspect `report.failed` and `report.failures`; set `RETRY_FAILED=True` on a later run to retry them. Another job holding the camera's lock raises `StreamBusyError`. A crash may repeat a model call, while stable Qdrant point IDs prevent duplicate vector rows.


In [ ]:
RETRY_FAILED = False
with load_homesight_from_config_file(config_path, env_file=env_path) as components:
    report: IndexingRunReport = components.analytics_service.analyze_stream(
        camera_name, limit=100, retry_failed=RETRY_FAILED
    )
    print(report.model_dump_json(indent=2))

## 7. Search footage and request playback locations

Edit `QUERY` to describe something in the recorded scene. Scores are larger-is-better, not probabilities. Results are checked against current catalog availability, model identity, and revision. The S3 extra signs playback locations for 600 seconds; regenerate them by rerunning this step after expiry. This step closes clients after the search while returned result models remain usable.

In the Qdrant dashboard, inspect the configured collection and its payloads. Use the library for text search: Qdrant itself does not create your model's text embeddings.


In [ ]:
QUERY = "person carrying a bag"
with load_homesight_from_config_file(config_path, env_file=env_path) as components:
    matches: list[SegmentSearchResult] = components.search_service.search_segments_by_text(
        QUERY, limit=5, camera_names=[camera_name], include_playback_url=True
    )
for match in matches:
    print(match.segment_id, match.start_time.isoformat(), match.similarity_score)
if not matches:
    print("No results: check available footage, indexing report, and configured model.")

## 8. Play a match

This asks your browser to fetch the first result's temporary S3 URL. Your browser must reach the configured S3 endpoint and support the clip's codec/container. An H.265 clip may require an external player even when native frame sampling succeeds. A playback URL is not a transcoding operation. Clear notebook outputs before sharing a session containing private footage or temporary URLs.


In [ ]:
if matches and matches[0].playback_url:
    display(Video(url=matches[0].playback_url, embed=False))
else:
    print("Run a successful search first.")

## 9. Inspect status and finish

All recorder/model/storage clients above are closed by their contexts. Uploaded media, the catalog, staging journals, cache, and Qdrant volume remain. Rerun recording to resume work, then index again; running only indexing does not record new footage.

Stop Qdrant with `docker compose stop qdrant`; resume with `docker compose up -d qdrant`. No cleanup or retention is run by this notebook. To review archive retention separately, use `uv run cli.py retention configs/tutorial.toml --dry-run --env-file .env`.

See the [user guide](../docs/user-guide.md), [model input selection](../docs/user-guide.md#video-or-sampled-frames), and [troubleshooting](../docs/user-guide.md#troubleshooting) for the full configuration and recovery rules.


In [ ]:
matches[0].playback_url

In [ ]:
print(catalog.get_stream_status(camera_name).model_dump_json(indent=2))
print("Session closed; retained data can be reused.")